# Homework Week 5

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
num = 100000 
 
difficulty = np.random.uniform(0, 1, (num,)) 
 
speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0) 
 
accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty + np.random.normal(0, 0.3, (num,)), 0), 1) 
 
df = pd.DataFrame({'difficulty': difficulty, 'speed': speed, 'accident': accident}) 

In [4]:
coefs = []
for _ in range(100): # 100 datasets
    difficulty = np.random.uniform(0, 1, (num,))
    speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0)
    # Regress speed (Y) on difficulty (X)
    X = sm.add_constant(difficulty)
    model = sm.OLS(speed, X)
    results = model.fit()
    coefs.append(results.params[1])
np.mean(coefs)

np.float64(-9.651897151233618)

In [6]:
coefs_xz = []
for _ in range(100):
    difficulty = np.random.uniform(0, 1, (num,))
    speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0)
    accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty + np.random.normal(0, 0.3, (num,)), 0), 1)
    
    # Regress speed (Y) on difficulty (X) and accident (Z)
    indep = sm.add_constant(np.column_stack((difficulty, accident)))
    model = sm.OLS(speed, indep)
    results = model.fit()
    coefs_xz.append(results.params[1]) # Index 1 is difficulty (X)

np.mean(coefs_xz)

np.float64(-10.329336990107059)

# Week 6

In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# 1. Load the dataset
df = pd.read_csv('homework_6.1.csv')

# Separate treated (X=1) and untreated (X=0) items
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# 2. Fit NearestNeighbors models based on the confounder Z
nn_treated = NearestNeighbors(n_neighbors=1).fit(treated[['Z']])
nn_untreated = NearestNeighbors(n_neighbors=1).fit(untreated[['Z']])

# 3. Find counterfactuals
# For treated items: closest untreated item
_, dist_treated = nn_untreated.kneighbors(treated[['Z']])
te_treated = treated['Y'] - untreated.iloc[dist_treated.flatten()]['Y'].values

# For untreated items: closest treated item
_, dist_untreated = nn_treated.kneighbors(untreated[['Z']])
te_untreated = treated.iloc[dist_untreated.flatten()]['Y'].values - untreated['Y']

# 4. Calculate Treatment Effects
# Overall Average Treatment Effect (ATE)
total_items = len(te_treated) + len(te_untreated)
ate = (te_treated.sum() + te_untreated.sum()) / total_items

# Average Treatment Effect on the Treated (ATT)
att = te_treated.mean()

# Average Treatment Effect on the Untreated (ATU)
atu = te_untreated.mean()

# Optimal treatment effect (max TE across all untreated)
opt_te = te_untreated.max()

print(f"ATE: {ate}")
print(f"ATT: {att}")
print(f"ATU: {atu}")
print(f"Optimal TE: {opt_te}")

ATE: 1.6952701427497883
ATT: 1.8464085071912946
ATU: 1.5494765534751722
Optimal TE: 2.172469885577008


# Week 7

In [1]:
import pandas as pd
import statsmodels.api as sm

# Load the data
df_71 = pd.read_csv("homework_7.1.csv")

def get_stratified_coef(w_val, delta=0.2):
    # Select a narrow window around the target W value
    subset = df_71[(df_71['W'] >= w_val - delta) & (df_71['W'] <= w_val + delta)]
    
    # Prepare predictors (X and Z) and add constant for intercept
    X_vars = subset[['X', 'Z']]
    X_vars = sm.add_constant(X_vars)
    y = subset['Y']
    
    # Fit OLS model
    model = sm.OLS(y, X_vars).fit()
    return model.params['X']

# Calculate coefficients at W = -1, 0, and 1
coef_m1 = get_stratified_coef(-1)
coef_0 = get_stratified_coef(0)
coef_1 = get_stratified_coef(1)

print(f"Coefficient of X at W ≈ -1: {coef_m1:.4f}")
print(f"Coefficient of X at W ≈ 0: {coef_0:.4f}")
print(f"Coefficient of X at W ≈ 1: {coef_1:.4f}")

Coefficient of X at W ≈ -1: 0.8699
Coefficient of X at W ≈ 0: 1.4177
Coefficient of X at W ≈ 1: 1.9343


In [2]:
import numpy as np
import statsmodels.api as sm

def make_error(corr_const, num):
    """Generates an autocorrelated error term."""
    err = list()
    prev = np.random.normal(0, 1)
    for n in range(num):
        # Update rule for autocorrelation
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1)
        err.append(prev)
    return np.array(err)

def run_autocorr_simulation(corr_const, num_pts=100, trials=500):
    estimates = []
    std_errors = []
    
    for _ in range(trials):
        # Treatment X and outcome Y both share the same error structure
        error_t = make_error(corr_const, num_pts)
        error_y = make_error(corr_const, num_pts)
        
        X = error_t 
        Y = 1.0 * X + error_y
        
        # OLS regression with intercept
        X_with_const = sm.add_constant(X)
        model = sm.OLS(Y, X_with_const).fit()
        
        estimates.append(model.params[1])
        std_errors.append(model.bse[1])
        
    sd_of_estimates = np.std(estimates)
    mean_of_se = np.mean(std_errors)
    return sd_of_estimates, mean_of_se

# Run for different correlation constants
for cc in [0.2, 0.5, 0.8]:
    sd_est, m_se = run_autocorr_simulation(cc)
    ratio = sd_est / m_se
    print(f"corr_const={cc}:")
    print(f"  (i) Empirical SD of estimates: {sd_est:.4f}")
    print(f"  (ii) Mean of reported SE:      {m_se:.4f}")
    print(f"  Ratio (i)/(ii):                {ratio:.4f}\n")

corr_const=0.2:
  (i) Empirical SD of estimates: 0.1058
  (ii) Mean of reported SE:      0.1008
  Ratio (i)/(ii):                1.0500

corr_const=0.5:
  (i) Empirical SD of estimates: 0.1363
  (ii) Mean of reported SE:      0.1009
  Ratio (i)/(ii):                1.3510

corr_const=0.8:
  (i) Empirical SD of estimates: 0.2567
  (ii) Mean of reported SE:      0.0998
  Ratio (i)/(ii):                2.5718



# Week 8

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

# 1. Load the data
df_81 = pd.read_csv("homework_8.1.csv")

# 2. Estimate propensity scores using logistic regression
# Z values predict X (treatment)
Z = df_81[['Z']]
X = df_81['X']

# Using sklearn for propensity score estimation
propensity_model = LogisticRegression()
propensity_model.fit(Z, X)

# Calculate P(X=1 | Z)
# predict_proba returns [P(X=0), P(X=1)]
df_81['propensity_score'] = propensity_model.predict_proba(Z)[:, 1]

# 3. Calculate inverse probability weights
# Weight = 1/P for X=1, and 1/(1-P) for X=0
df_81['ip_weight'] = np.where(df_81['X'] == 1, 
                              1 / df_81['propensity_score'], 
                              1 / (1 - df_81['propensity_score']))

# 4. Estimate the ATE
# ATE = Mean(Y * weight | X=1) - Mean(Y * weight | X=0)
# But standard ATE via IPW is (1/N) * sum( X*Y/P - (1-X)*Y/(1-P) )
# or more simply, the difference of weighted means
weighted_y_treat = (df_81[df_81['X'] == 1]['Y'] * df_81[df_81['X'] == 1]['ip_weight']).sum() / len(df_81)
weighted_y_control = (df_81[df_81['X'] == 0]['Y'] * df_81[df_81['X'] == 0]['ip_weight']).sum() / len(df_81)

ate_ipw = weighted_y_treat - weighted_y_control

print(f"Propensity Score Head:\n{df_81[['X', 'Z', 'propensity_score', 'ip_weight']].head()}\n")
print(f"Estimated ATE using IPW: {ate_ipw:.4f}")

Propensity Score Head:
   X         Z  propensity_score  ip_weight
0  1  1.764052          0.840114   1.190315
1  0  0.400157          0.584646   2.407585
2  0  0.978738          0.711082   3.461195
3  0  2.240893          0.892793   9.327719
4  1  1.867558          0.853089   1.172211

Estimated ATE using IPW: 2.2393


C:\Users\santo\AppData\Local\Temp\ipykernel_13680\2693009216.py:20: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_81['propensity_score'] = propensity_model.predict_proba(Z)[:, 1]
C:\Users\santo\AppData\Local\Temp\ipykernel_13680\269300921

In [4]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import mahalanobis

# 1. Load the dataset
df_82 = pd.read_csv("homework_8.2.csv")

# 2. Prepare the inverse covariance matrix
# Use all Z1 and Z2 values, make them into a 2xN matrix (Z1 and Z2 as rows)
# In pandas, this is Z1 and Z2 columns. df[['Z1', 'Z2']].values.T gives 2xN
z_matrix = df_82[['Z1', 'Z2']].values.T
# Find the 2x2 covariance and invert
cov_matrix = np.cov(z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# 3. Separate treated and untreated
treated = df_82[df_82['X'] == 1].copy()
untreated = df_82[df_82['X'] == 0].copy()

def find_nearest_mahalanobis(t_row, u_pool, i_cov):
    z_t = t_row[['Z1', 'Z2']].values
    distances = u_pool.apply(lambda u_row: mahalanobis(z_t, u_row[['Z1', 'Z2']].values, i_cov), axis=1)
    # Match to the single nearest untreated item (index of minimum distance)
    return distances.idxmin()

# 4. Perform matching with replacement
# For each treated item, find the index of the nearest untreated neighbor
treated['match_idx'] = treated.apply(lambda r: find_nearest_mahalanobis(r, untreated, inv_cov), axis=1)

# 5. Calculate results (e.g., ATT)
treated['Y_counterfactual'] = untreated.loc[treated['match_idx'], 'Y'].values
treated['treatment_effect'] = treated['Y'] - treated['Y_counterfactual']

att_mahalanobis = treated['treatment_effect'].mean()

print(f"Inverse Covariance Matrix:\n{inv_cov}\n")
print(f"First 5 Matches:\n{treated[['X', 'Y', 'match_idx', 'Y_counterfactual']].head()}\n")
print(f"Average Treatment Effect on the Treated (ATT): {att_mahalanobis:.4f}")

Inverse Covariance Matrix:
[[ 2.02734407 -1.03387633]
 [-1.03387633  1.06684813]]

First 5 Matches:
   X         Y  match_idx  Y_counterfactual
0  1  4.652085        681          0.428954
1  1  2.590221        752         -0.034844
2  1  3.898981        892          1.164988
3  1  5.857179        922          1.797450
4  1  3.647489        922          1.797450

Average Treatment Effect on the Treated (ATT): 3.4377


C:\Users\santo\AppData\Local\Temp\ipykernel_13680\285755188.py:31: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  treated['Y_counterfactual'] = untreated.loc[treated['match_idx'], 'Y'].values
C:\Users\santo\AppData\Local\Temp\ipykernel_13680\

In [6]:
from scipy.spatial.distance import cdist

# 1. Vectorized calculation for distances between all treated and untreated
treated_z = treated[['Z1', 'Z2']].values
untreated_z = untreated[['Z1', 'Z2']].values

# Using cdist to find distances between all pairs
all_dists = cdist(treated_z, untreated_z, metric='mahalanobis', VI=inv_cov)

# 2. Get nearest neighbor distance for each treated row
min_dists_per_treated = np.min(all_dists, axis=1)

# 3. Find the index where the "nearest neighbor distance" is maximum
outlier_idx = np.argmax(min_dists_per_treated)
max_dist = min_dists_per_treated[outlier_idx]

# 4. Extract Z values for the treated item and its closest match
treated_outlier = treated.iloc[outlier_idx]
match_idx_in_untreated = np.argmin(all_dists[outlier_idx])
untreated_match = untreated.iloc[match_idx_in_untreated]

print(f"Treated item with least common support (Index {treated_outlier.name}):")
print(f"  Z1: {treated_outlier['Z1']:.4f}, Z2: {treated_outlier['Z2']:.4f}")
print(f"  Smallest distance to control group: {max_dist:.4f}")
print(f"\nNearest untreated neighbor (Index {untreated_match.name}):")
print(f"  Z1: {untreated_match['Z1']:.4f}, Z2: {untreated_match['Z2']:.4f}")


Treated item with least common support (Index 494):
  Z1: 2.6962, Z2: 0.5382
  Smallest distance to control group: 1.3830

Nearest untreated neighbor (Index 418):
  Z1: 1.5200, Z2: -1.2822
